# European Steel Decarbonization ⚙️🏭💨♻️
<img src="../images/steel_industry.jpg" alt="Industrial Steel Production" width="100%">
<p style="font-size: 0.8em; margin-top: 4px;">
  <a href="https://unsplash.com/de/fotos/rasenplatz-SQEo2zSXB04" target="_blank">Photo by Anton Eprev</a>
</p>

## Exploratory Data Analysis (EDA)

**Team:** SuSteelAible  
**Author:** Irene Polgar<br>
**Date:** December 2025

**Purpose:** 
This notebook explores emission patterns in the European steel industry (2013-2024) as part of the SuSteelAible capstone project, examining how technology, scale, and carbon pricing relate to emission intensity trends.

### Data Sources

**Company-Level Dataset (Primary Analysis)**
- 13 major European steel companies (2013-2024)
- Technology coverage: BF-BOF (blast furnace) and EAF (electric arc furnace)
- Metrics: Production volumes, emissions (Scope 1+2), emission intensity

**Industry Benchmarks (Comparison)**
- EU emission data from European Environment Agency (EEA) (verified emisions from the steel sector)
- EU production data from Statista/Eurostat
- Global emission and production data from World Steel Association

### Analysis Approach
This exploratory analysis examines:
1. Emission intensity patterns by technology
2. Time trends and stability
3. Geographic and operational factors
4. The relationship between carbon pricing and emissions

---

## 1. Setup & Data Loading

In [ ]:
# Add scripts folder to Python path
import sys
from pathlib import Path

scripts_path = Path.cwd().parent / 'scripts'
sys.path.insert(0, str(scripts_path))

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.patches import Patch

# Using data_loader script for consistent loading across notebooks.
# This handles: Excel file reading, apostrophe encoding fixes, year-type conversion, filtering steps
# See scripts/data_loader.py for full implementation
from data_loader import (
    load_company_data,
    load_eu_data,
    load_global_data,
    filter_complete_data,
    print_data_summary
)

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Libraries loaded successfully")

In [ ]:
# Load data

# Load full dataset (all companies including Asian comparisons)
df_all = load_company_data(fix_apostrophes=True)
print(f"✓ All companies: {df_all['company'].nunique()} companies, {len(df_all)} observations")

# Load European companies only (for main analysis)
df = load_company_data(fix_apostrophes=True, filter_region='Europe')
print(f"✓ European companies: {df['company'].nunique()} companies, {len(df)} observations")

# Load EU-wide and global context data
eu_data = load_eu_data()
global_data = load_global_data()
print(f"✓ EU-wide data: {len(eu_data)} years")
print(f"✓ Global trend data: {len(global_data)} years")

### 1.1 Dataset Structure

In [ ]:
df.head()

**Key variables:**

- **Identifiers:** `company`, `country`, `technology`, `year`
- **Production:** `production` (Mt/year) - crude steel production volume
- **Emissions:** 
  - `scope1` (Mt CO₂e) - Direct emissions from steel production
  - `scope2_location` (Mt CO₂e) - Indirect emissions from electricity (grid average)
  - `scope2_market` (Mt CO₂e) - Indirect emissions from electricity (actual contracts)
- **Emission Intensity:** `intensity_location_co2e` (tCO₂e/t) - Emissions per tonne of steel (scope 1)

### 1.2 Understanding Steel Emissions Metrics

#### Emission Scopes
Our analysis focuses on **operational emissions** under direct company control:

- **Scope 1**: Direct emissions from steel-making processes (blast furnaces, electric arc furnaces, on-site combustion)
- **Scope 2**: Indirect emissions from purchased electricity used in production

**Why not Scope 3?** 
Value chain emissions (Scope 3) are excluded because reporting is inconsistent across companies and methodologies vary significantly.

#### Emission Intensity
**Emission intensity** (tCO₂e/t) is calculated as:
`Intensity = (Scope 1 + Scope 2) / Production Volume`

This metric accounts for differences in company size by normalizing emissions to production output. It allows direct comparison between a 1 Mt/year minimill and a 10 Mt/year integrated steel plant.

#### Scope 2 Methodologies
Companies report Scope 2 using two approaches:
- **Location-based**: Uses average grid emissions (consistent benchmark)
- **Market-based**: Uses actual energy contracts (shows renewable investments)

**Analysis approach varies by section:**
- **Time series analysis** (Section 5.2):  Uses **"best available"** - selects whichever Scope 2 method (location or market-based) has more years of data per company to maximize time series length
- **Cross-company comparisons** (Sections 7, 8): Uses **location-based only** for consistency

### 1.3 Analysis Roadmap

| Section | Analysis Focus | Research Question | Data Source | Metric Used |
|---------|----------------|-------------------|-------------|-------------|
| **4** | Technology comparison | How do BF-BOF vs EAF emissions differ? | Company data | **Scope 1 intensity** |
| **5.1** | Direct emission trends | Are direct emissions improving over time? | Company data + benchmarks | **Scope 1 intensity** |
| **5.2** | Operational emission trends | Are operational emissions improving over time?| Company data + benchmarks | **Total intensity (Scope 1+2)**<br>*best available method* |
| **6** | Scale effects | How does emission intensity vary with production scale? | Company data | **Scope 1 intensity** |
| **7** | Electricity intensity | Which companies have the lowest electricity emissions? | Company data | **Scope 2 intensity**<br>*location-based* |
| **8** | Carbon pricing impact | Do carbon price increases correspond to emission reductions? | **EU ETS data** | **EU-wide intensity**<br>*location-based* |

---

## 2. Data Quality Checks

Validation of data integrity before analysis:
- Data types verification
- Missing values analysis
- Duplicate detection
- Value range validation
- Internal consistency checks

In [ ]:
# 2.1 Data types verification

print("\nDATA TYPES:")
print("-"*80)

# Check critical column types
type_checks = {
    'company': 'object',
    'country': 'object',
    'technology': 'object',
    'year': ['int64', 'int32'],
    'production': ['float64', 'float32'],
    'scope1': ['float64', 'float32'],
    'scope2_location': ['float64', 'float32'],
    'scope2_market': ['float64', 'float32']
}

all_correct = True
for col, expected_type in type_checks.items():
    actual_type = str(df[col].dtype)
    if isinstance(expected_type, list):
        is_correct = actual_type in expected_type
        expected_display = ' or '.join(expected_type)
    else:
        is_correct = actual_type == expected_type
        expected_display = expected_type
    
    status = "✓" if is_correct else "✗"
    print(f"  {status} {col:20s}: {actual_type:10s} (expected: {expected_display})")
    if not is_correct:
        all_correct = False

if all_correct:
    print("\n✓ All data types correct")
else:
    print("\n⚠️  Some data types need attention")


### 2.2 Missing Values

**Expected missingness patterns:**

The dataset has known data gaps that reflect changing reporting practices and data availability:

- **Scope 1 emissions:** 
  - ArcelorMittal (European division): separate emissions data not available (only emissions intensity)
  - Salzgitter AG (2013-2015): Not publicly available

- **Market-based Scope 2 emissions:** Increasingly reported in recent years, lacking for earlier years (except Outokumpu)

- **Production data:** Minimal gaps (e.g., SIDENOR 2019)

- **Intensity metrics:** Separately reported only by some companies, calculated in section 3.2 from available production and emissions data

In [ ]:
print("\nMISSING VALUES:")
print("-"*80)

missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(1)
missing_summary = pd.DataFrame({
    'Count': missing_count,
    'Percentage': missing_pct
})

# Show only columns with missing data
missing_summary = missing_summary[missing_summary['Count'] > 0].sort_values('Count', ascending=False)

if len(missing_summary) > 0:
    print(missing_summary.to_string())
    print(f"\nMost missing: {missing_summary.index[0]} ({missing_summary.iloc[0]['Percentage']:.1f}%)")
else:
    print("✓ No missing values found")

print("\n✓ Missing data patterns documented above")

In [ ]:
# 2.3 Duplicates check

print("\nDUPLICATES:")
print("-"*80)

duplicates = df.duplicated(subset=['company', 'year']).sum()
print(f"Duplicate company-year combinations: {duplicates}")

if duplicates == 0:
    print("✓ No duplicates found (each company-year appears once)")
else:
    print(f"⚠️  Found {duplicates} duplicate rows!")
    dup_mask = df.duplicated(subset=['company', 'year'], keep=False)
    print("\nDuplicate rows:")
    print(df[dup_mask][['company', 'year']].sort_values(['company', 'year']))

### 2.4 Value Range Validation

**Expected ranges:**
- **Production:** 0.1-100 Mt/year (varies by company size)
- **Scope 1:** 0.1-200 Mt CO₂e (varies by technology and scale)
- **Scope 2:** 0-0.2 Mt CO₂e (can be negative if renewable energy credits or on-site generation are exceeding consumption)
- **Emission Intensity:** 0.1-2.5 tCO₂e/t

Negative Scope 2 values are legitimate in GHG accounting when companies generate more renewable energy than they consume.

In [ ]:
print("Value Ranges:")
print("-"*70)

# Production ranges
print(f"Production (Mt):        {df['production'].min():6.2f} to {df['production'].max():6.2f}")
print(f"Scope 1 (Mt CO₂e):      {df['scope1'].min():6.2f} to {df['scope1'].max():6.2f}")
print(f"Scope 2 loc (Mt CO₂e):  {df['scope2_location'].min():6.2f} to {df['scope2_location'].max():6.2f}")
print(f"Scope 2 mar (Mt CO₂e):  {df['scope2_market'].min():6.2f} to {df['scope2_market'].max():6.2f}")

if 'intensity_location_co2e' in df.columns:
    intensity_df = df[df['intensity_location_co2e'].notna()]
    print(f"Intensity (tCO₂e/t):    {intensity_df['intensity_location_co2e'].min():6.2f} to {intensity_df['intensity_location_co2e'].max():6.2f}")

# Validation checks
print("\nValidation:")
print(f"  ✓ All production values positive: {not (df['production'] <= 0).any()}")
print(f"  ✓ All Scope 1 values non-negative: {not (df['scope1'] < 0).any()}")

negative_scope2 = (df['scope2_location'] < 0).sum()
if negative_scope2 > 0:
    companies = df[df['scope2_location'] < 0]['company'].unique()
    print(f"  ℹ️  Negative Scope 2: {negative_scope2} observations ({', '.join(companies)})")
    print(f"     Verified as legitimate (renewable energy generation)")
else:
    print("  ✓ All Scope 2 values non-negative")

### 2.5 Internal Consistency Check

Validate that provided intensity values match calculated values from components (see section 1.5 for calculation).

**Expected differences:**
- **<0.001:** Perfect match (rounding only)
- **0.001-0.05:** Acceptable (rounding/conversion differences)
- **>0.05:** Larger difference due to boundary adjustments or fiscal/calendar conversions

In [ ]:
if 'intensity_location_co2e' in df.columns:
    df_check = df[df['intensity_location_co2e'].notna()].copy()
    
    # Calculate intensity from components
    df_check['calculated_intensity'] = (
        (df_check['scope1'] + df_check['scope2_location']) / df_check['production']
    )
    
    # Calculate difference
    df_check['intensity_diff'] = abs(
        df_check['intensity_location_co2e'] - df_check['calculated_intensity']
    )
    
    print("Intensity Validation:")
    print("-"*70)
    print(f"Max difference:  {df_check['intensity_diff'].max():.6f} tCO₂e/t")
    print(f"Mean difference: {df_check['intensity_diff'].mean():.6f} tCO₂e/t")
    
    # Categorize differences
    perfect = (df_check['intensity_diff'] < 0.001).sum()
    good = (df_check['intensity_diff'] < 0.05).sum()
    moderate = ((df_check['intensity_diff'] >= 0.05) & (df_check['intensity_diff'] < 0.15)).sum()
    
    print(f"\n  ✓ Perfect match (<0.001):  {perfect}/{len(df_check)}")
    print(f"  ✓ Good match (<0.05):      {good}/{len(df_check)} ({good/len(df_check)*100:.1f}%)")
    
    if moderate > 0:
        print(f"  ℹ️  Larger differences:      {moderate} observations")
        print("\n  Companies with differences >0.05:")
        for company in df_check[df_check['intensity_diff'] >= 0.05]['company'].unique():
            max_diff = df_check[df_check['company'] == company]['intensity_diff'].max()
            print(f"    • {company:30s} (max: {max_diff:.4f})")
else:
    print("No pre-calculated intensity column found")

print("\n✓ All discrepancies verified against source reports")

---

## 3. Data Preparation & Metric Calculation

Filter to companies with sufficient data and calculate derived metrics:
- Filter companies with ≥4 years of data
- Calculate emission intensity
- Create technology groupings

### 3.1 Filter companies with few datapoints

In [ ]:
# Filter to companies with at least 4 years of Scope 1 data
df_complete = filter_complete_data(df, min_years=4)

print(f"After filtering: {df_complete['company'].nunique()} companies, {len(df_complete)} observations\n")

# List companies
print("Companies included:")
for company in sorted(df_complete['company'].unique()):
    n_years = len(df_complete[df_complete['company'] == company])
    year_range = f"{df_complete[df_complete['company']==company]['year'].min()}-{df_complete[df_complete['company']==company]['year'].max()}"
    print(f"  • {company:30s} {n_years:2d} years ({year_range})")


### 3.2 Calculate Emissions Intensities

The code below calculates three key metrics:
- `scope1_intensity`: Direct emission intensity per tonne
- `scope2_intensity_best`: Best available Scope 2 method per company
- `total_intensity_best`: Combined Scope 1 + Scope 2

**"Best available" selection strategy:**  
For each company, we compare the number of years with location-based vs. market-based Scope 2 data and select whichever method provides more complete coverage.

In [ ]:
# Scope 1 intensity (direct emissions per tonne)
df_complete['scope1_intensity'] = df_complete['scope1'] / df_complete['production']

# Scope 2 intensity (both methods)
df_complete['scope2_intensity_location'] = df_complete['scope2_location'] / df_complete['production']
df_complete['scope2_intensity_market'] = df_complete['scope2_market'] / df_complete['production']

print("Scope 2 data availability:")
print(f"  • Location-based: {df_complete['scope2_intensity_location'].notna().sum()} observations")
print(f"  • Market-based:   {df_complete['scope2_intensity_market'].notna().sum()} observations")

# Select best available method per company
company_methods = []
for company in df_complete['company'].unique():
    company_data = df_complete[df_complete['company'] == company]
    n_location = company_data['scope2_intensity_location'].notna().sum()
    n_market = company_data['scope2_intensity_market'].notna().sum()
    
    company_methods.append({
        'company': company,
        'total_years': len(company_data),
        'location_available': n_location,
        'market_available': n_market,
        'use_method': 'location' if n_location >= n_market else 'market',
        'usable_obs': max(n_location, n_market)
    })

method_df = pd.DataFrame(company_methods).sort_values('usable_obs', ascending=False)

# Display methodology selection
print(f"\nMethodology selection by company:")
print(f"{'Company':<30} {'Total':<8} {'Location':<10} {'Market':<10} {'Use':<10}")
print("-"*80)
for _, row in method_df.iterrows():
    print(f"{row['company']:<30} {row['total_years']:<8} "
          f"{row['location_available']:<10} {row['market_available']:<10} "
          f"{row['use_method']:<10}")

print(f"\n✓ Location-based: {(method_df['use_method']=='location').sum()} companies")
print(f"✓ Market-based:   {(method_df['use_method']=='market').sum()} companies")

# Create best available intensity column
df_complete['scope2_intensity_best'] = df_complete.apply(
    lambda row: (
        row['scope2_intensity_location'] 
        if method_df[method_df['company']==row['company']]['use_method'].iloc[0] == 'location'
        else row['scope2_intensity_market']
    ),
    axis=1
)

# Calculate total intensity (Scope 1 + best Scope 2)
df_complete['total_intensity_best'] = (
    df_complete['scope1_intensity'] + df_complete['scope2_intensity_best']
)
df_complete['total_intensity_location'] = (
    df_complete['scope1_intensity'] + df_complete['scope2_intensity_location']
)

print("\n✓ Emission intensity metrics calculated")

### 3.3 Simplified Technology Grouping

Create a simplified technology column for a more balanced sample distribution.

In [ ]:
# Simplified technology categories
df_complete['technology_group'] = df_complete['technology'].apply(
    lambda x: 'EAF' if 'EAF' in str(x) 
    else 'BF-BOF' if 'BF-BOF' in str(x)
    else 'Other'
)

print("Technology distribution:")
tech_dist = df_complete.groupby('technology_group').agg({
    'company': 'nunique',
    'year': 'count'
}).rename(columns={'company': 'Companies', 'year': 'Observations'})
print(tech_dist)

In [ ]:
# Define consistent color scheme for technology types
# Used in Sections 4, 6, and 7 for visual consistency
tech_colors = {
    'BF-BOF': '#e41a1c',           # Red (carbon-intensive)
    'EAF': '#4daf4a',              # Green (cleaner)
    'EAF Stainless': '#377eb8'     # Blue (stainless steel)
}

def get_tech_color(tech):
    """
    Assign color based on technology string matching.
    
    Handles variations in technology naming (e.g., 'EAF Stainless', 
    'BF-BOF → H₂-DRI') by checking if keywords are present in the string.
    
    Parameters:
    -----------
    tech : str
        Technology name (e.g., 'BF-BOF', 'EAF', 'EAF Stainless')
    
    Returns:
    --------
    str : Hex color code
    """
    # Check for BF-BOF first (most specific)
    if 'BF-BOF' in tech:
        return tech_colors['BF-BOF']
    # Check for Stainless (more specific than general EAF)
    elif 'Stainless' in tech:
        return tech_colors['EAF Stainless']
    # Check for EAF (general electric arc furnace)
    elif 'EAF' in tech:
        return tech_colors['EAF']
    # Fallback (shouldn't happen with clean data)
    else:
        return '#999999'  # Gray

print("✓ Technology color scheme defined")
print(f"  Colors: {len(tech_colors)} technology types")

---

## 4. Emission Distribution by Technology

How does emission intensity vary by production technology?

In [ ]:
# Statistics by technology

stats_by_tech = df_complete.groupby('technology')['scope1_intensity'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('median', 'median'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max')
]).round(3)

print("Summary Statistics")
print("-"*70)
print("Scope 1 Intensity by Technology:")
print(stats_by_tech)

# Save
stats_by_tech.to_csv('../results/EDA/stats_by_technology.csv')
print("\n✓ Saved: stats_by_technology.csv")

In [ ]:
# Boxplot: Emission intensity by technology

fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')

# Prepare data
plot_data = df_complete[df_complete['scope1_intensity'].notna()]
technologies = sorted(plot_data['technology'].unique())

# Create boxplot
bp = ax.boxplot([
    plot_data[plot_data['technology'] == tech]['scope1_intensity'].dropna()
    for tech in technologies
], labels=technologies, patch_artist=True)

# Color boxes using shared color scheme
for patch, tech in zip(bp['boxes'], technologies):
    patch.set_facecolor(get_tech_color(tech))
    patch.set_alpha(0.7)

# Formatting
ax.set_ylabel('Scope 1 Intensity (tCO$_2$e/t steel)', fontsize=12, fontweight='bold')
ax.set_title('Scope 1 Emission Intensity by Technology', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(labelsize=12, pad=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/EDA/01_boxplot_technology.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: 01_boxplot_technology.png")


**Key Observations:** 

- **Technology drives emissions:** Clear separation between technology groups, with BF-BOF (~1.4-2.5 tCO₂e/t) significantly higher than EAF technologies

- **EAF spectrum:** Standard EAF (~0.08-0.13 tCO₂e/t) shows lowest emissions, while EAF Stainless (~0.35-0.53 tCO₂e/t) is higher due to energy-intensive stainless production, but still ~75% lower than BF-BOF

- **BF-BOF outliers:** Wider range with four high outliers (>2.0 tCO₂e/t) belonging to Acciaierie d'Italia (see below)

---

## 5. Company-Level Emissions Profiles

This section analyses emission trends over time for individual companies, comparing them against EU and global steel industry averages.

In [ ]:
# Prepare Benchmark Data

# Align EU and global data to common years
common_years = sorted(set(eu_data['year']) & set(global_data['year']))
eu_aligned = eu_data[eu_data['year'].isin(common_years)].sort_values('year').reset_index(drop=True).copy()
global_aligned = global_data[global_data['year'].isin(common_years)].sort_values('year').reset_index(drop=True).copy()

# Force numeric conversion
eu_aligned['year'] = pd.to_numeric(eu_aligned['year'], errors='coerce')
eu_aligned['eu_intensity_external'] = pd.to_numeric(eu_aligned['eu_intensity_external'], errors='coerce')
global_aligned['year'] = pd.to_numeric(global_aligned['year'], errors='coerce')
global_aligned['co2_intensity_harmonized_tco2_per_t'] = pd.to_numeric(global_aligned['co2_intensity_harmonized_tco2_per_t'], errors='coerce')

# Drop NaN
eu_aligned = eu_aligned.dropna(subset=['year', 'eu_intensity_external'])
global_aligned = global_aligned.dropna(subset=['year', 'co2_intensity_harmonized_tco2_per_t'])

# Extract numpy arrays
eu_years = eu_aligned['year'].values
eu_intensity = eu_aligned['eu_intensity_external'].values
global_years = global_aligned['year'].values
global_intensity = global_aligned['co2_intensity_harmonized_tco2_per_t'].values

print("✓ Benchmark data prepared")
print(f"  Common years: {eu_years[0]:.0f}-{eu_years[-1]:.0f} ({len(eu_years)} years)")

In [ ]:
# Helper Function: Time Series Plotting

def plot_intensity_timeseries(df, intensity_col, ylabel, title, filename):
    """
    Plot emission intensity time series with EU/Global benchmark comparison.
    
    This function creates a standardized time series visualization showing:
    - Individual company trends (colored lines)
    - EU average benchmark (blue dotted line with shading)
    - Global average benchmark (pink dotted line with shading)
    
    Parameters:
    -----------
    df : pd.DataFrame
        Complete dataset with company-level data
    intensity_col : str
        Column name for intensity metric to plot (e.g., 'scope1_intensity')
    ylabel : str
        Y-axis label text
    title : str
        Plot title
    filename : str
        Output filename for saving the plot
    
    Returns:
    --------
    None (saves plot and displays it)
    """
    
    # Prepare data: filter to rows with valid intensity values
    df_plot = df[df[intensity_col].notna()].copy()
    companies = sorted(df_plot['company'].unique())
    
    # Create distinct color for each company (using seaborn's tab20 palette)
    palette = sns.color_palette("tab20", n_colors=len(companies))
    color_map = dict(zip(companies, palette))
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6), facecolor='white')
    
    # Plot benchmark data
    ax.fill_between(eu_years, 0, eu_intensity, 
                    color="#C6E5F4", alpha=0.2, zorder=1)
    ax.fill_between(global_years, eu_intensity, global_intensity, 
                    color="#E3C3C3", alpha=0.2, zorder=1)
    ax.plot(eu_years, eu_intensity, 
            linestyle=':', color="#60ABF1", linewidth=2.5, 
            label='EU Average', zorder=5)
    ax.plot(global_years, global_intensity, 
            linestyle=':', color="#D2AAAE", linewidth=3, 
            label='Global Average', zorder=5)
    
    # Plot each company's trend
    for company in companies:
        company_data = df_plot[df_plot['company'] == company].sort_values('year')
        ax.plot(company_data['year'], company_data[intensity_col],
                marker='o', label=company, linewidth=2.5, markersize=6,
                color=color_map.get(company, '#7F8C8D'),
                zorder=10)
    
    # Formatting
    ax.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), 
              frameon=True, fancybox=True, shadow=True, 
              fontsize=11, framealpha=0.95)
    
    ax.tick_params(axis='both', labelsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Save and display
    plt.tight_layout()
    plt.savefig(f'../results/EDA/{filename}', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

### 5.1 Time Series: Scope 1 Intensity (European Companies)

Examining **direct emission intensity** for individual companies.

In [ ]:
# Plot Scope 1 intensity (direct operational emissions)
plot_intensity_timeseries(
    df=df_complete,
    intensity_col='scope1_intensity',
    ylabel='Direct Emission Intensity (Scope 1, tCO$_2$e/t)',
    title='Direct Emission Intensity: European Companies vs EU/Global Averages',
    filename='02_scope1_timeseries_european.png'
)

**Key Observations:**
- European companies span the full technology range from EAF (~0.2-0.6) to BF-BOF (~1.4-2.0)
- Most companies show relatively flat trends
- Our dataset focuses on major producers (predominantly BF-BOF technology)
- The lower EU average (~0.7 tCO₂e/t) reflects the presence of many smaller EAF producers across Europe which are not captured in our dataset

### 5.2 Operational Emission Intensity (Scope 1+2)
Examining **total operational intensity** (Scope 1 + Scope 2), using "best available" Scope 2 data (see Section 3.2).

In [ ]:
# Plot operational intensity (Scope 1 + Scope 2)
plot_intensity_timeseries(
    df=df_complete,
    intensity_col='total_intensity_best',
    ylabel='Operational Intensity (Scope 1+2, tCO$_2$e/t)',
    title='Operational Emission Intensity: European Companies vs EU/Global Averages',
    filename='03_scope12_timeseries_european.png'
)

In [ ]:
# Statistics by technology (operational intensity)

stats_by_tech = df_complete.groupby('technology')['total_intensity_best'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('median', 'median'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max')
]).round(3)

print("Summary Statistics")
print("-"*70)
print("Operational Intensity by Technology:")
print(stats_by_tech)

**Key Observations:**
- Similar pattern (technology clusters) as in 5.1 , though Scope 2 adds variability (e.g. Acerinox EU shows relatively high intensity)
- Some EAF companies (Outokumpu, Acerinox EU, Sidenor) show declining trend
- This suggests declining emissions intensity is driven by grid decarbonization rather than Scope 1 improvements

---

## 6. Production Volume vs Emission Intensity

Does production scale affect emissions efficiency? This scatter plot examines the relationship between production volume and direct emission intensity across different steel production technologies.

Each point represents one company-year observation (2013-2024).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')

# Plot by detailed technology (using shared color scheme)
for tech in sorted(df_complete['technology'].dropna().unique()):
    tech_df = df_complete[df_complete['technology'] == tech]
    
    ax.scatter(
        tech_df['production'], 
        tech_df['scope1_intensity'],
        label=tech, s=80, alpha=0.6, 
        color=get_tech_color(tech),  # Uses shared color function
        edgecolors='black', linewidth=0.5
    )

# Formatting
ax.set_xlabel('Production (Mt/year)', fontsize=12, fontweight='bold')
ax.set_ylabel('Direct Emission Intensity (Scope 1, tCO$_2$e/t)', fontsize=12, fontweight='bold')
ax.set_title('Production Volume vs Direct Emission Intensity by Technology', 
             fontsize=14, fontweight='bold', pad=12)
ax.legend(title='Technology', fontsize=11, framealpha=0.9, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/EDA/04_production_vs_intensity.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 04_production_vs_intensity.png")

**Key Observation:** Technology choice (BF-BOF vs EAF) drives emissions intensity more than production scale. No clear scale effect visible within technology groups.

---

## 7. Electricity Emission Intensity by Company (Scope 2)

Which companies have the lowest/highest electricity-related emissions?

Here, we use location-based methodology for consistent cross-company comparison.

*Note: Data for Outokumpu limited to 2020-2022.*

In [ ]:
# Calculate average Scope 2 intensity by company (location-based for consistency)
scope2_summary = df_complete[
    df_complete['scope2_intensity_location'].notna()
].groupby(['company', 'technology'])['scope2_intensity_location'].mean().reset_index()
scope2_summary = scope2_summary.sort_values('scope2_intensity_location')

# Assign colors using shared color scheme
colors = [get_tech_color(tech) for tech in scope2_summary['technology']]

# Create plot
fig, ax = plt.subplots(figsize=(10, 7), facecolor='white')

ax.barh(
    scope2_summary['company'], 
    scope2_summary['scope2_intensity_location'],
    color=colors, alpha=0.7, edgecolor='black', linewidth=0.5
)

ax.set_xlabel('Average Scope 2 Intensity (tCO$_2$e/t), location-based', fontsize=12, fontweight='bold')
ax.set_ylabel('Company', fontsize=12, fontweight='bold')
ax.set_title('Electricity Emissions Intensity by Company',
             fontsize=14, fontweight='bold', pad=12)
ax.grid(True, alpha=0.3, axis='x')

# Add legend
legend_elements = [
    Patch(facecolor=tech_colors['BF-BOF'], label='BF-BOF', alpha=0.7),
    Patch(facecolor=tech_colors['EAF'], label='EAF', alpha=0.7),
    Patch(facecolor=tech_colors['EAF Stainless'], label='EAF Stainless', alpha=0.7)
]
ax.legend(title='Technology', fontsize=11, handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../results/EDA/05_scope2_by_company.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 05_scope2_by_company.png")

In [ ]:
# Summary statistics
print("\nScope 2 Intensity by Technology (location-based):")
print("-"*70)
tech_scope2 = df_complete.groupby('technology')['scope2_intensity_location'].agg(
    ['count', 'mean', 'median', 'std']
).round(3)
print(tech_scope2)

**Key Observations:**
- EAF companies show higher Scope 2 intensity due to electricity-intensive melting process
- BF-BOF companies have lower Scope 2 intensity (primarily use coal/coke for heat)
- Grid crbon intensity presumably plays an additional role (varies by country)

---

## 8. Carbon Pricing and Emission Trends

How have EU steel emissions responded to carbon pricing policy over the past decade?

This section examines the relationship between EU ETS carbon prices and emission intensity trends in the European steel sector (2013-2023).

**Data:**

- **Carbon prices:** EU Emissions Trading System (EU ETS) market data
  - The EU ETS is a cap-and-trade system requiring steel producers to purchase allowances for each tonne of CO₂ emitted
  - Prices reflect market supply/demand for emission permits

- **Emission intensity:** European Environment Agency (EEA) Emissions Trading Viewer
  - Verified emissions data from ~240 iron and steel installations across the EU

In [ ]:
# Use EU-wide data from EEA (verified emissions)
eu_carbon = eu_data[['year', 'carbon_price', 'eu_intensity_external']].copy().sort_values('year')

# Create dual-axis plot
fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
ax2 = ax.twinx()

# Left axis: Carbon price (red)
line1 = ax.plot(eu_carbon['year'], eu_carbon['carbon_price'], 
               'o-', linewidth=3, markersize=10, color='#e41a1c', 
               label='EU ETS Carbon Price', zorder=3)
ax.fill_between(eu_carbon['year'], eu_carbon['carbon_price'], 
                alpha=0.2, color='#e41a1c', zorder=1)

# Right axis: EU-wide emissions intensity (blue)
line2 = ax2.plot(eu_carbon['year'], eu_carbon['eu_intensity_external'], 
                's--', linewidth=2.5, markersize=8, color='#377eb8', 
                label='EU Steel Intensity (EEA)', zorder=2)

# Formatting
ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('EU ETS Carbon Price (EUR/tonne CO$_2$)', fontsize=12, fontweight='bold', color='#e41a1c')
ax2.set_ylabel('EU Steel Intensity (tCO$_2$e/t)', fontsize=12, fontweight='bold', color='#377eb8')
ax.set_title('The Carbon Price Paradox:\nRising Carbon Prices, Stable Emissions', 
            fontsize=14, fontweight='bold', pad=15)
ax2.set_ylim(0, 2.0)

ax.tick_params(axis='y', labelcolor='#e41a1c', labelsize=11)
ax2.tick_params(axis='y', labelcolor='#377eb8', labelsize=11)
ax.grid(True, alpha=0.3, zorder=0)

# Combined legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper left', fontsize=10, framealpha=0.95)

plt.tight_layout()
plt.savefig('../results/EDA/07_carbon_price_paradox.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved: 07_carbon_price_paradox.png")

**Key Observations:**
- Carbon prices increased 17-fold (€5 → €85+) while emission intensity remained stable (~0.7-0.8 tCO₂e/t), illustrating the "carbon price paradox"
- This stability reflects technology lock-in: existing steel plants cannot easily switch production methods in response to price signals
- Decarbonization requires technology transitions (plant retirements, new EAF/H₂-DRI capacity), not just carbon pricing

---

## 9. Key Findings

**Technology choice drives emissions:** EAF companies produce 70-80% lower intensity (~0.2-0.5 tCO₂e/t) compared to BF-BOF (~1.5-2.0 tCO₂e/t). The technology gap between EAF and BF-BOF is larger than all other observed differences.

**The carbon price paradox:** Despite carbon prices rising 17-fold from 2013 to 2023, emission intensity across the EU steel sector remained stable. This suggests strong technology lock-in - existing plants can't easily switch production methods in response to price signals alone.

**Electricity matters for EAF:** EAF companies show substantially higher Scope 2 emissions due to their electricity-intensive production process, while BF-BOF plants rely primarily on fossil fuels for heat.

**Scale doesn't matter:** Production volume shows no clear relationship with emission intensity - a small plant can be just as clean (per tonne) as a large one.

---

### Next Steps

This exploratory analysis provides the foundation for:
- **Baseline modeling:** Technology gap analysis using decision tree framework
- **Action score analysis:** Statistical trend testing and performance scoring
- **Integration:** Linking quantitative trends with sustainability reporting narratives